In [4]:
keywords = [
    "atrial fibrillation",
    "type 2 diabetes mellitus",
    "pediatric asthma management",
    "acute otitis media",
    "chronic kidney disease",
    "iron deficiency anemia",
    "community acquired pneumonia",
    "gestational diabetes",
    "celiac disease diagnosis",
    "allergic rhinitis treatment"
]

In [12]:
import requests
import xml.etree.ElementTree as ET

BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

def search_pubmed(keyword, max_results=5):
    url = f"{BASE}esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": keyword,
        "sort": "pub date",
        "retmax": max_results,
        "retmode": "json"
    }

    response = requests.get(url, params=params).json()
    pmids = response["esearchresult"]["idlist"]

    return pmids

def fetch_pubmed_details(pmids):
    url = f"{BASE}efetch.fcgi"
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml"
    }

    xml_data = requests.get(url, params=params).text
    root = ET.fromstring(xml_data)

    results = []

    for article in root.findall(".//PubmedArticle"):
        pmid = article.findtext(".//PMID")

        title = article.findtext(".//ArticleTitle")

        abstract_parts = [ "".join(el.itertext()) for el in article.findall(".//AbstractText")
            if "".join(el.itertext()).strip()
        ]
        abstract = " ".join(abstract_parts) if abstract_parts else None
        if abstract is None: continue

        results.append({
            "pmid": pmid,
            "content": title + abstract,
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        })

    return results

In [6]:
import time
article_map = {}

for keyword in keywords:
    print(keyword)
    pmids = search_pubmed(keyword, 10)
    article_map[keyword] = pmids
    time.sleep(0.5)

atrial fibrillation
type 2 diabetes mellitus
pediatric asthma management
acute otitis media
chronic kidney disease
iron deficiency anemia
community acquired pneumonia
gestational diabetes
celiac disease diagnosis
allergic rhinitis treatment


In [7]:
article_map_final = {}

for key, art_ids in article_map.items():
    for art_id in art_ids:
        if art_id in article_map_final:
            article_map_final[art_id]["keywords"].append(key)
        else:
            article_map_final[art_id] = {"keywords": [key]}

In [13]:
articles = fetch_pubmed_details(article_map_final.keys())

In [14]:
articles

[{'pmid': '41520281',
  'content': 'Comparative insights into the gut-heart axis: cross-species and cross-population perspectives.Gut microbiota research has rapidly expanded our understanding of host-microbe interactions in cardiovascular diseases, yet translation of these insights remains challenged by species-specific differences and substantial population heterogeneity. In this review, we synthesize current evidence across rodents, swine, non-human primates, and multi-ethnic human cohorts to delineate conserved versus context-dependent features of the gut-heart axis. Rodent models remain indispensable for mechanistic discovery, enabling causal testing through germ-free, antibiotic-treated, and humanized microbiota platforms, whereas large-animal models better replicate human cardiac anatomy, physiology, and microbial ecology. Human studies provide essential clinical relevance, demonstrating that patients with myocardial infarction, coronary artery disease, atrial fibrillation, and 

In [15]:
import json
with open("corpus.json", 'w', encoding="utf-8") as f:
    json.dump(articles, f, ensure_ascii=False)